# DXY Signal Evaluation
`apr_jan_mapped_w_macro.csv` — Apr 2025 + Jan 2026 articles

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/dec_jan_mapped_w_macro.csv")

relevant = df[df["is_relevant"] == True].copy()
critical = df[df["is_critical"] == True].copy()
dir_crit = critical[critical["direction"].isin(["up","down"])].copy()

horizons = ["pct_5m", "pct_15m", "pct_1h", "pct_4h", "pct_1d"]
sd = {h: df[h].std() for h in horizons}

print(f"Total articles : {len(df):,}")
print(f"Relevant       : {len(relevant):,}  ({len(relevant)/len(df)*100:.1f}%)")
print(f"Critical       : {len(critical):,}  ({len(critical)/len(relevant)*100:.1f}% of relevant)")
print(f"Critical w/ up/down direction: {len(dir_crit):,}")
print()
print("SDs  5m={:.4f}%  15m={:.4f}%  1h={:.4f}%  4h={:.4f}%  1d={:.4f}%".format(
    sd["pct_5m"], sd["pct_15m"], sd["pct_1h"], sd["pct_4h"], sd["pct_1d"]))

Total articles : 499
Relevant       : 235  (47.1%)
Critical       : 80  (34.0% of relevant)
Critical w/ up/down direction: 73

SDs  5m=0.0315%  15m=0.0466%  1h=0.0813%  4h=0.1783%  1d=0.3648%


## 1. Directional Accuracy by Criticality Level
Filtered to rows where direction ∈ {up, down} and intraday price was matched.

In [3]:
def dir_accuracy(subset, horizon):
    s = subset[subset["direction"].isin(["up","down"]) & subset[horizon].notna()].copy()
    actual = s[horizon].apply(lambda x: "up" if x > 0 else "down")
    hits = (actual == s["direction"]).sum()
    return hits, len(s)

levels = ["high", "medium", "low"]
rows = []
for lvl in levels + ["all_critical"]:
    sub = dir_crit if lvl == "all_critical" else critical[
        (critical["criticality_level"] == lvl) & critical["direction"].isin(["up","down"])]
    row = {"criticality": lvl}
    for h in horizons:
        hits, n = dir_accuracy(sub, h)
        row[h] = f"{hits/n*100:.1f}% ({hits}/{n})" if n else "n/a"
    rows.append(row)

display(pd.DataFrame(rows).set_index("criticality"))

,pct_5m,pct_15m,pct_1h,pct_4h,pct_1d
criticality,,,,,
high,54.5% (18/33),42.4% (14/33),21.9% (7/32),40.6% (13/32),42.3% (11/26)
medium,50.0% (16/32),56.2% (18/32),50.0% (16/32),67.9% (19/28),58.6% (17/29)
low,n/a,n/a,n/a,n/a,n/a
all_critical,52.3% (34/65),49.2% (32/65),35.9% (23/64),53.3% (32/60),50.9% (28/55)


## 2. Directional Accuracy by Direction Confidence

In [4]:
rows = []
for conf in ["high", "medium", "low"]:
    sub = dir_crit[dir_crit["direction_confidence"] == conf]
    row = {"dir_confidence": conf, "n": len(sub)}
    for h in horizons:
        hits, n = dir_accuracy(sub, h)
        row[h] = f"{hits/n*100:.1f}% ({hits}/{n})" if n else "n/a"
    rows.append(row)

display(pd.DataFrame(rows).set_index("dir_confidence"))

,n,pct_5m,pct_15m,pct_1h,pct_4h,pct_1d
dir_confidence,,,,,,
high,26,60.9% (14/23),52.2% (12/23),18.2% (4/22),43.5% (10/23),36.8% (7/19)
medium,37,45.5% (15/33),48.5% (16/33),36.4% (12/33),53.3% (16/30),58.6% (17/29)
low,10,55.6% (5/9),44.4% (4/9),77.8% (7/9),85.7% (6/7),57.1% (4/7)


## 3. Table Usage
Critical articles that used the historical response table vs. didn't.

In [5]:
tbl = critical.groupby("table_used").agg(n=("id","count"),
    pct_5m_mean=("pct_5m","mean"), pct_15m_mean=("pct_15m","mean"),
    pct_1h_mean=("pct_1h","mean"), pct_4h_mean=("pct_4h","mean"),
).round(4)
tbl.index = tbl.index.map({True: "table_used", False: "no_table"})
display(tbl)

print()
for label, mask in [("table_used", critical["table_used"]==True), ("no_table", critical["table_used"]==False)]:
    sub = critical[mask & critical["direction"].isin(["up","down"])]
    for h in ["pct_5m", "pct_15m", "pct_1h"]:
        hits, n = dir_accuracy(sub, h)
        acc = f"{hits/n*100:.1f}% ({hits}/{n})" if n else "n/a"
        print(f"  {label}  {h}: {acc}")
    print()

,n,pct_5m_mean,pct_15m_mean,pct_1h_mean,pct_4h_mean
table_used,,,,,
no_table,64,-0.0014,-0.0038,-0.0058,0.0167
table_used,16,0.0059,-0.0025,-0.0183,-0.0158



  table_used  pct_5m: 50.0% (8/16)
  table_used  pct_15m: 50.0% (8/16)
  table_used  pct_1h: 50.0% (8/16)

  no_table  pct_5m: 53.1% (26/49)
  no_table  pct_15m: 49.0% (24/49)
  no_table  pct_1h: 31.2% (15/48)



## 4. Mean |DXY %| Change by Criticality Level

In [6]:
rows = []
for lvl in levels + ["all_relevant", "all"]:
    if lvl == "all_relevant":
        sub = relevant[relevant["pct_1h"].notna()]
    elif lvl == "all":
        sub = df[df["pct_1h"].notna()]
    else:
        sub = df[(df["criticality_level"] == lvl) & df["pct_1h"].notna()]
    row = {"group": lvl, "n": len(sub)}
    for h in horizons:
        row[f"|{h}|"] = round(sub[h].abs().mean(), 4)
    rows.append(row)

display(pd.DataFrame(rows).set_index("group"))

,n,|pct_5m|,|pct_15m|,|pct_1h|,|pct_4h|,|pct_1d|
group,,,,,,
high,32,0.0326,0.0521,0.0745,0.1764,0.2752
medium,39,0.0149,0.0251,0.0510,0.1151,0.2895
low,134,0.0168,0.0303,0.0615,0.1181,0.2683
all_relevant,205,0.0189,0.0327,0.0615,0.1273,0.2736
all,390,0.0196,0.0327,0.0599,0.1290,0.2816


## 5. Moves ≥ 1SD / 2SD Captured in High & Medium Critical

In [7]:
def sd_capture(subset, horizon, threshold):
    s = subset[subset["direction"].isin(["up","down"]) & subset[horizon].notna()].copy()
    s["actual_dir"] = s[horizon].apply(lambda x: "up" if x > 0 else "down")
    above = s[s[horizon].abs() >= threshold]
    hits  = (above["actual_dir"] == above["direction"]).sum()
    return hits, len(above)

rows = []
for lvl in ["high", "medium"]:
    sub = critical[critical["criticality_level"] == lvl]
    for h in horizons:
        h1, n1 = sd_capture(sub, h, sd[h])
        h2, n2 = sd_capture(sub, h, sd[h]*2)
        rows.append({
            "criticality": lvl,
            "horizon":     h,
            "1SD thresh":  f"{sd[h]:.4f}%",
            "≥1SD":        f"{h1}/{n1} = {h1/n1*100:.1f}%" if n1 else "n/a",
            "2SD thresh":  f"{sd[h]*2:.4f}%",
            "≥2SD":        f"{h2}/{n2} = {h2/n2*100:.1f}%" if n2 else "n/a",
        })

display(pd.DataFrame(rows).set_index(["criticality","horizon"]))

1SD thresh          ≥1SD 2SD thresh          ≥2SD
criticality horizon                                                  
high        pct_5m     0.0315%  6/10 = 60.0%    0.0629%   4/6 = 66.7%
            pct_15m    0.0466%  5/12 = 41.7%    0.0933%   4/7 = 57.1%
            pct_1h     0.0813%  4/13 = 30.8%    0.1626%   2/5 = 40.0%
            pct_4h     0.1783%  6/13 = 46.2%    0.3565%   1/3 = 33.3%
            pct_1d     0.3648%   3/8 = 37.5%    0.7296%    0/1 = 0.0%
medium      pct_5m     0.0315%   1/2 = 50.0%    0.0629%           n/a
            pct_15m    0.0466%   2/4 = 50.0%    0.0933%           n/a
            pct_1h     0.0813%  4/4 = 100.0%    0.1626%  1/1 = 100.0%
            pct_4h     0.1783%   4/5 = 80.0%    0.3565%  1/1 = 100.0%
            pct_1d     0.3648%   2/5 = 40.0%    0.7296%    0/2 = 0.0%

## 6. Mann-Whitney U Test: Critical vs Non-Critical Absolute Moves
One-sided test (H₁: critical articles produce larger absolute DXY moves).

In [8]:
from scipy.stats import mannwhitneyu

rows = []
for h in horizons:
    crit_vals    = critical[h].dropna().abs()
    noncrit_vals = df[df["is_critical"] != True][h].dropna().abs()
    stat, p = mannwhitneyu(crit_vals, noncrit_vals, alternative="greater")
    rows.append({
        "horizon":        h,
        "n_critical":     len(crit_vals),
        "n_non_critical": len(noncrit_vals),
        "mean |crit|":    round(crit_vals.mean(), 4),
        "mean |non-crit|":round(noncrit_vals.mean(), 4),
        "U stat":         round(stat, 1),
        "p-value":        round(p, 4),
        "sig (p<0.05)":   "yes" if p < 0.05 else "no",
    })

mw_df = pd.DataFrame(rows).set_index("horizon")
display(mw_df)

,n_critical,n_non_critical,mean |crit|,mean |non-crit|,U stat,p-value,sig (p<0.05)
horizon,,,,,,,
pct_5m,72,329,0.0228,0.0188,12968.0,0.1020,no
pct_15m,72,326,0.0368,0.0313,12684.5,0.1413,no
pct_1h,71,319,0.0616,0.0596,11547.5,0.3978,no
pct_4h,67,292,0.1475,0.1254,11057.5,0.0480,yes
pct_1d,60,256,0.2885,0.2828,7556.0,0.5775,no
